# Survey Stimulus Generation — Attestation Trust Study

**AI assistance disclosure:** This stimulus-generation tooling was built with the assistance of an AI assistant (Claude) for the code scaffolding (parsing, validation, file output) and tutoring how to use the API and prompt correctly. The generation prompt, the attestation-display wording, and all curation decisions are the author's own (Harry Staley). Per the study's documented method, an LLM generates candidate stimuli which the author then curates. Use of generative AI follows the CS 6795 course policy.

In [40]:
from __future__ import annotations

import json
import re
import time
from collections import Counter
from typing import List, Tuple, TypedDict
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import display
from dotenv import load_dotenv
from datetime import datetime
load_dotenv()
if not load_dotenv():
    print("WARNING: .env not found; run setup_env.py to create it.")
print(f"Key loaded: {load_dotenv()}")
print("NOTE: Be sure that you have a .env file with your OpenAI API key.")

Key loaded: True
NOTE: Be sure that you have a .env file with your OpenAI API key.


In [41]:
MODEL: str = "gpt-5.5"            # model name
# NOTE: Temperature is not available in gpt-5.5, but it is in others.
# TEMPERATURE: float = 0.7        # controls randomness; higher = more varied output
MAX_RETRIES: int = 5            # how many times to retry on hard failure
LENGTH_RATIO_WARN: float = 0.25 # warn if answers differ >25% in length
SENTENCE_DIFF_WARN: int = 1     # warn if sentence counts differ by >1

In [42]:
class Stem(TypedDict):
    """Schema for one generated survey stimulus."""
    stem_id: int
    stakes: str
    category: str
    consequence_type: str
    fact_structure: str
    topic: str
    question_text: str
    correct_answer: str
    incorrect_answer: str
    source_name: str
    source_citation: str
    source_url: str
    ground_truth_note: str

# Verifies that the JSON object has the required keys as defined in the Stem class.
REQUIRED_KEYS: set[str] = {
    "stem_id", "stakes", "category", "consequence_type", "fact_structure",
    "topic", "question_text",
    "correct_answer", "incorrect_answer", "source_name",
    "source_citation", "source_url", "ground_truth_note",
}

In [43]:
prompt_template = Path("generation_prompt.md").read_text(encoding="utf-8")
GENERATION_PROMPT = prompt_template.format(
    schema_fields=", ".join(Stem.__annotations__)
)

In [44]:
def attestation_text(att_level: str, item: Stem) -> str:
    """Return the rendered attestation display for a given attestation level."""
    if att_level == "none":
        return ""
    if att_level == "weak":
        return f"Source: {item['source_name']} — {item['source_citation']}"
    return (f"Source: {item['source_name']} — {item['source_citation']}\n"
            f"Publisher verified ({item['source_name']})\n"
            f"Document unaltered since publication\n"
            f"Independently checked for relevance")

In [45]:
def parse_json(raw_text: str) -> List[Stem]:
    """Parse model output into stems; recover the [...] array if wrapped."""
    raw_text = raw_text.strip()
    raw_text = re.sub(r"^```(?:json)?|```$", "", raw_text, flags=re.MULTILINE).strip()
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        start, end = raw_text.find("["), raw_text.rfind("]") + 1
        if start == -1 or end == 0:
            raise
        return json.loads(raw_text[start:end])

In [46]:
def sentence_count(text: str) -> int:
    """Rough sentence count, robust to decimals/abbreviations (for warnings only)."""
    if not text.strip():
        return 0
    t = re.sub(r"\d+\.\d+", "0", text)
    for abbr in ("Dr.", "Mr.", "Mrs.", "Ms.", "U.S.", "U.K.", "e.g.", "i.e.",
                 "etc.", "mg.", "mL.", "vs.", "Inc.", "Ltd.", "Fig.", "No."):
        t = t.replace(abbr, abbr.replace(".", ""))
    return max(len(re.findall(r"[.!?]+", t)), 1)

In [47]:
def validate_stems(items: List[Stem]) -> Tuple[List[str], List[str]]:
    """Return (errors, warnings). Errors trigger retry; warnings flag for curation."""
    errors: List[str] = []
    warnings: List[str] = []

    if len(items) != 12:
        errors.append(f"Expected 12 stems, found {len(items)}.")
    low = sum(x.get("stakes") == "low" for x in items)
    high = sum(x.get("stakes") == "high" for x in items)
    if low != 6 or high != 6:
        errors.append(f"Expected 6 low / 6 high; found {low} low / {high} high.")
    if sorted(x.get("stem_id", -1) for x in items) != list(range(1, 13)):
        errors.append("stem_id values must be 1..12 with no gaps/dupes.")

    topics: List[str] = []
    for item in items:
        sid = item.get("stem_id", "?")
        if set(item.keys()) != REQUIRED_KEYS:
            errors.append(f"Stem {sid} schema mismatch.")
            continue
        topics.append(item["topic"].lower())
        ca, ia = item["correct_answer"], item["incorrect_answer"]
        if ia.strip() == "REFUSED_NEEDS_MANUAL" or not ia.strip():
            errors.append(f"Stem {sid} incorrect_answer REFUSED -- build manually.")
            continue
        if abs(sentence_count(ca) - sentence_count(ia)) > SENTENCE_DIFF_WARN:
            warnings.append(f"Stem {sid}: sentence-count mismatch (review).")
        if max(len(ca), len(ia)) and abs(len(ca) - len(ia)) / max(len(ca), len(ia)) > LENGTH_RATIO_WARN:
            warnings.append(f"Stem {sid}: answer-length mismatch (review).")

    dups = [t for t, c in Counter(topics).items() if c > 1]
    if dups:
        errors.append(f"Duplicate topics: {dups}")
    return errors, warnings

In [48]:
# Test the whole logic chain with fake data -- no API, no cost.
_mock = [
    {"stem_id": i, "stakes": "low" if i <= 6 else "high",
     "category": "Consumer/retail facts" if i <= 6 else "Financial penalty/loss",
     "consequence_type": "financial",
     "fact_structure": "threshold-amount",
     "topic": f"topic{i}",
     "question_text": "Q?", "correct_answer": "A true statement here.",
     "incorrect_answer": "A false statement here.", "source_name": "Src",
     "source_citation": "Doc, src.org", "source_url": "https://src.org",
     "ground_truth_note": "note"}
    for i in range(1, 13)
]
_errors, _warnings = validate_stems(_mock)
print("errors:", _errors)
print("warnings:", _warnings)
assert not _errors, "mock should pass structural validation"
print("MOCK PASSED — logic chain works.")

errors: []
warnings: []
MOCK PASSED — logic chain works.


In [49]:
def generate_response() -> List[Stem]:
    """One generation call; stem_ids are assigned in code, not trusted from the model."""
    response = client.responses.create(
        model=MODEL,
        input=GENERATION_PROMPT,
    )
    items = parse_json(response.output_text)
    # assign stem_ids by position -- the model is unreliable at sequential numbering
    for i, item in enumerate(items, start=1):
        item["stem_id"] = i
    return items

In [50]:
client = OpenAI()
_test = client.responses.create(
    model=MODEL,
    input='Return exactly this JSON and nothing else: [{"ok": 1}]',
)
print(repr(_test.output_text))

'[{"ok": 1}]'


In [51]:
def generate_with_retries() -> Tuple[List[Stem], List[str]]:
    """Generate stems; retry on HARD failures, surface warnings for curation."""
    last_errors: List[str] = []
    for attempt in range(1, MAX_RETRIES + 1):
        print("=" * 80)
        print(f"ATTEMPT {attempt}/{MAX_RETRIES}")
        try:
            items = generate_response()
            errors, warnings = validate_stems(items)
            if not errors:
                print("Structural validation passed.")
                if warnings:
                    print(f"\n{len(warnings)} item(s) flagged for human curation:")
                    for w in warnings:
                        print("  ~", w)
                else:
                    print("No curation warnings.")
                print()
                return items, warnings
            print("Hard failures (regenerating):")
            for e in errors:
                print("  -", e)
            last_errors = errors
        except Exception as e:
            print("Generation failed:", str(e))
            last_errors = [str(e)]
        time.sleep(1)
    raise RuntimeError(
        "Generation failed after retries. Last hard failures:\n"
        + "\n".join(last_errors)
        + "\n\nIf failures are REFUSED incorrect answers on sensitive items, "
          "generate the rest and CONSTRUCT those items manually (document it)."
    )


In [52]:
def build_loop_table(items: List[Stem]) -> pd.DataFrame:
    """Expand validated stems into the 72-row Qualtrics loop table."""
    rows: List[dict] = []
    for item in items:
        for att_level in ("none", "weak", "strong"):
            for correctness in ("correct", "incorrect"):
                answer = (item["correct_answer"] if correctness == "correct"
                          else item["incorrect_answer"])
                rows.append({
                    "stem_id": item["stem_id"],
                    "stakes": item["stakes"],
                    "question_text": item["question_text"],
                    "answer_text": answer,
                    "att_level": att_level,
                    "correctness": correctness,
                    "attestation_text": attestation_text(att_level, item),
                })
    return pd.DataFrame(rows)

In [53]:
items, warnings = generate_with_retries()

stems_df = pd.DataFrame(items)
loop_df = build_loop_table(items)

# show results
print("\n" + "=" * 80 + "\nGENERATED STEMS\n" + "=" * 80)
print(stems_df.to_string(index=False))
print("\n" + "=" * 80 + "\nQUALTRICS LOOP TABLE\n" + "=" * 80)
print(loop_df.to_string(index=False))

display(stems_df)
display(loop_df)

# write outputs to a dedicated directory
out = Path("output")
out.mkdir(exist_ok=True)
ts = datetime.now().strftime("%Y-%m-%dT%H%M%S")

stems_df.to_csv(out / f"generated_stems_{ts}.csv", index=False)
loop_df.to_csv(out / f"qualtrics_loop_table_{ts}.csv", index=False)
with open(out / f"generated_stems_{ts}.json", "w", encoding="utf-8") as f:
    json.dump(items, f, indent=2)

metadata = {
    "model": MODEL,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "max_retries": MAX_RETRIES,
    "curation_warnings": warnings,
    "note": "Soft warnings indicate items flagged for human curation "
            "(matched length/sentence count), per the documented method.",
}
with open(out / f"generation_metadata_{ts}.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nSaved to {out}/: generated_stems_{ts}.json, generated_stems_{ts}.csv, "
      f"qualtrics_loop_table_{ts}.csv, generation_metadata_{ts}.json")
if warnings:
    print(f"\n{len(warnings)} item(s) need curation review (see {out}/generation_metadata_{ts}.json).")

ATTEMPT 1/5
Structural validation passed.
No curation warnings.


GENERATED STEMS
 stem_id stakes                                                                                                                                                                                                           category                                     consequence_type                fact_structure                                     topic                                                                                                                                                                            question_text                                                                                                                       correct_answer                                                                                                                 incorrect_answer                             source_name                                                                              so

,stem_id,stakes,category,consequence_type,fact_structure,topic,question_text,correct_answer,incorrect_answer,source_name,source_citation,source_url,ground_truth_note
0,1,high,"Financial penalty/loss — tax penalties, excise...",federal tax penalty,threshold/amount,traditional IRA early distributions,"At the federal level, if an adult takes a nonq...",The distribution is generally subject to a 10%...,The distribution is generally subject to a 5% ...,Internal Revenue Service,"IRS Tax Topic No. 557, Additional Tax on Early...",https://www.irs.gov/taxtopics/tc557,The IRS states that early IRA distributions be...
1,2,high,"Benefit/coverage forfeiture — Medicare, COBRA,...",coverage delay or late-enrollment penalty,deadline/window,Medicare Part B initial enrollment,For someone first eligible for Medicare at 65 ...,It is a seven-month period that starts three m...,It is a six-month period that starts two month...,Medicare.gov,"Medicare.gov, When can I sign up for Medicare?",https://www.medicare.gov/basics/get-started-wi...,Medicare describes the initial enrollment peri...
2,3,high,Consumer-protection deadlines — fraud-reportin...,increased liability for unauthorized transfers,deadline-linked liability cap,lost or stolen debit card reporting,"Under federal electronic fund transfer rules, ...",Their loss is generally limited to no more tha...,Their loss is generally limited to no more tha...,Consumer Financial Protection Bureau,"CFPB Ask CFPB, What should I do if my ATM or d...",https://www.consumerfinance.gov/ask-cfpb/what-...,The CFPB explains that reporting a lost or sto...
3,4,high,Legal-right or claim forfeiture — statute of l...,loss of right to petition before paying assess...,deadline,Tax Court petition deadline,"At the federal level, how many days does a tax...",They generally have 90 days from the notice da...,They generally have 60 days from the notice da...,United States Tax Court,"United States Tax Court, Starting a Case",https://www.ustaxcourt.gov/petitioners_start.html,The U.S. Tax Court states that a deficiency pe...
4,5,high,Food safety,foodborne illness risk,safety threshold,reheating leftovers,"When reheating leftovers, what internal temper...","Leftovers should be reheated to 165°F, as meas...","Leftovers should be reheated to 145°F, as meas...",USDA Food Safety and Inspection Service,"USDA FSIS, Leftovers and Food Safety",https://www.fsis.usda.gov/food-safety/safe-foo...,USDA FSIS instructs consumers to reheat leftov...
5,6,high,Cybersecurity,identity theft or tax fraud exposure,authentication rule,IRS contact methods,"At the federal level, how does the IRS general...",The IRS generally first contacts taxpayers by ...,The IRS generally first contacts taxpayers by ...,Internal Revenue Service,"IRS, How to know it’s really the IRS calling o...",https://www.irs.gov/newsroom/how-to-know-its-r...,The IRS says it generally first contacts taxpa...
6,7,low,Consumer/retail facts — standard return window...,loss of a small cancellation refund,threshold/entitlement,federal cooling-off rule for home sales,For a qualifying sale made at a buyer’s home i...,The federal rule generally applies to qualifyi...,The federal rule generally applies to qualifyi...,Federal Trade Commission,"FTC Consumer Advice, Buyer’s Remorse: The FTC’...",https://consumer.ftc.gov/articles/buyers-remor...,The FTC explains that the Cooling-Off Rule gen...
7,8,low,Everyday financial-convenience facts — how lon...,minor delay in access to deposited funds,availability amount,check deposit funds availability,For a check deposited into a checking account ...,At least the first $225 is generally available...,At least the first $100 is generally available...,Consumer Financial Protection Bureau,"CFPB Ask CFPB, How quickly can I get money aft...",https://www.consumerfinance.gov/ask-cfpb/how-q...,"The CFPB states that, for many check deposits,..."
8,9,low,Shipping/postal facts — standard delivery time...,minor postage cost or mailing delay,weight threshold,First-Cla

,stem_id,stakes,question_text,answer_text,att_level,correctness,attestation_text
0,1,high,"At the federal level, if an adult takes a nonq...",The distribution is generally subject to a 10%...,none,correct,
1,1,high,"At the federal level, if an adult takes a nonq...",The distribution is generally subject to a 5% ...,none,incorrect,
2,1,high,"At the federal level, if an adult takes a nonq...",The distribution is generally subject to a 10%...,weak,correct,Source: Internal Revenue Service — IRS Tax Top...
3,1,high,"At the federal level, if an adult takes a nonq...",The distribution is generally subject to a 5% ...,weak,incorrect,Source: Internal Revenue Service — IRS Tax Top...
4,1,high,"At the federal level, if an adult takes a nonq...",The distribution is generally subject to a 10%...,strong,correct,Source: Internal Revenue Service — IRS Tax Top...
...,...,...,...,...,...,...,...
67,12,low,"In a baking weight chart, about how many grams...",One cup of all-purpose flour is listed as 140 ...,none,incorrect,
68,12,low,"In a baking weight chart, about how many grams...",One cup of all-purpose flour is listed as 120 ...,weak,correct,Source: King Arthur Baking — King Arthur Bakin...
69,12,low,"In a baking weight chart, about how many grams...",One cup of all-purpose flour is listed as 140 ...,weak,incorrect,Source: King Arthur Baking — King Arthur Bakin...
70,12,low,"In a baking weight chart, about how many grams...",One cup of all-purpose flour is listed as 120 ...,strong,correct,Source: King Arthur Baking — King Arthur Bakin...



Saved to output/: generated_stems_2026-06-21T190012.json, generated_stems_2026-06-21T190012.csv, qualtrics_loop_table_2026-06-21T190012.csv, generation_metadata_2026-06-21T190012.json
